**Pemrosesan Bahasa Alami - Marchell Adi Pratama (672023081)**


---


Sebagai tahap awal sebelum melakukan prapemrosesan data, proses ekstraksi data (scraping) dilakukan menggunakan Apify. Platform ini menyediakan layanan antarmuka pemrograman aplikasi (API) untuk mengakuisisi data dari berbagai platform digital, termasuk X (Twitter). Kumpulan data mentah (raw dataset) yang dihasilkan dari tahapan ini dapat ditinjau pada tautan berikut: [Dataset](https://drive.google.com/file/d/12UodEInqDM6AfNkxONxwML8EQzNYAq41/view?usp=sharing)

0.1. Install Library

In [1]:
!pip install apify-client pandas nltk Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.2/141.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.9 MB/s eta 0:00:00


0.2. Import Library & Download Resource NLTK untuk Tokenisasi dan **Stopwords**

In [2]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Download resource NLTK (Ditambah punkt_tab untuk mengatasi LookupError)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

1. Membaca Data CSV

In [3]:
nama_file = 'dataset x.csv'
df_raw = pd.read_csv(nama_file)
print(f"Berhasil memuat {len(df_raw)} baris data!")
display(df_raw)

Berhasil memuat 100 baris data!


,id,text,username,created_at,favorite_count,retweet_count,reply_count,view_count,lang
0,2062137699210059799,Kejaksaan Agung menetapkan Eks Kepala Badan Gi...,kumparan,Wed Jun 03 11:41:54 +0000 2026,460,210,161,36779.0,in
1,2061480968138162633,MAKAN BERGIZI GRATIS\nMakanan nyang bergizi na...,dzakira_af16440,Mon Jun 01 16:12:17 +0000 2026,0,0,0,4.0,in
2,2061944410942878129,Budalkan sam @JoeTirtoDiputro GA lokal malang ...,rumahalinsyiroh,Tue Jun 02 22:53:51 +0000 2026,7,5,2,1958.0,in
3,2061480809840955525,"Deungon ta pajoh makanan njang meugizi, tubôh ...",dzakira_af16440,Mon Jun 01 16:11:40 +0000 2026,0,0,0,1.0,ht
4,2062055925310259285,Kepala Kantor Staf Presiden (KSP) Dudung Abdur...,Beritasatu,Wed Jun 03 06:16:58 +0000 2026,39,8,23,16262.0,in
...,...,...,...,...,...,...,...,...,...
95,2062145972738908221,Tim Penyidik Jampidsus Kejaksaan Agung resmi m...,triwul82,Wed Jun 03 12:14:47 +0000 2026,8,3,4,398.0,in
96,2062141558250770769,Kejaksaan Agung menetapkan mantan pimpinan BGN...,detikcom,Wed Jun 03 11:57:14 +0000 2026,5,1,4,2225.0,in
97,2061789120238952688,"Tinjau MBG di SMPN 111 Jakarta Barat, Prabowo:...",pikiran_rakyat,Tue Jun 02 12:36:47 +0000 2026,3,0,1,1164.0,in
98,2061736980598358146,Presiden Prabowo Subianto melakukan pengecekan...,Metro_TV,Tue Jun 02 09:09:36 +0000 2026,2,1,1,618.0,in


In [4]:
# Menampilkan full text dari text saja
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
display(df_raw['text'])

,text
0,"Kejaksaan Agung menetapkan Eks Kepala Badan Gizi Nasional (BGN) Dadan Hindayana sebagai tersangka kasus dugaan penyimpangan tata kelola program Makan Bergizi Gratis (MBG). Dadan kini ditahan di Rutan Salemba.\n\nSelain Dadan, Kejagung juga menetapkan Eks Waka BGN Sonny Sonjaya dan https://t.co/OL4KIBw2Js"
1,"MAKAN BERGIZI GRATIS\nMakanan nyang bergizi nakeuh hak mandum ureung, dan deungon program makan gratis nyoe, geutanyoe jeut tapeu pasti mandum ureung meurumpok nutrisi nyang dibutuhkan untuk udep sehat.#PenuhiGiziIndonesia #Indonesiaemas #AstaCita #Indonesiamaju #GiziBaik https://t.co/oLpDVbSayE"
2,"Budalkan sam @JoeTirtoDiputro GA lokal malang kota, warung makan gratis, in-sya Allah buka jam 7.30 menyediakan kuang lebih 100 porsi, semoga Allah mudahkan dan jaga istiqomah kami semua. Barokallahu fiikum. \n#warungmakangratis\n#gratistanpasyarat https://t.co/SNddiK8uMa"
3,"Deungon ta pajoh makanan njang meugizi, tubôh geutanjoë leubèh teuga dan siap tahadapi tantangan, njan keuh sabab program makan gratis njan that peunténg untôk tapeugot kualitas hudép generasi bansa. #PenuhiGiziIndonesia #Indonesiaemas #AstaCita #Indonesiamaju #GiziBaik https://t.co/pDc1OpnGW6"
4,Kepala Kantor Staf Presiden (KSP) Dudung Abdurachman mengungkapkan bahwa penggeledahan kantor Badan Gizi Nasional (BGN) telah direncanakan sejak lama akibat adanya dugaan ketimpangan dalam program makan bergizi gratis. Beliau juga menyebut bahwa temuan dari berbagai sumber https://t.co/PltG01Mcod
5,"Badan Gizi Nasional katanya mau kasih makan bergizi gratis buat rakyat.\n\nTernyata yang kenyang duluan adalah kantong Dadan Hindayana &amp; kawan-kawan.\n\nProgram MBG bukan Makan Bergizi Gratis, tapi Makan Duit Bergizi — Rp49,5 M dari sertifikasi halal doang udah habis, sisanya entah https://t.co/O0JifHIhD7"
6,Eks Kepala BGN Dadan Hindayana ditetapkan sebagai tersangka dugaan korupsi penyimpangan tata kelola Makan Bergizi Gratis (MBG).\n\n~RS https://t.co/H1eCFfkbpc
7,"Evaluasi terhadap pelaksanaan Program Makan Bergizi Gratis (MBG) disebut menjadi salah satu pertimbangan Presiden Prabowo Subianto dalam mengganti pimpinan Badan Gizi Nasional (BGN).\n\nPresiden Prabowo telah menerima berbagai laporan, masukan, dan hasil evaluasi pelaksanaan https://t.co/0uvQFX3DH8"
8,Program Makan Bergizi Gratis ditolak sekolah https://t.co/OvsOXIQj2S
9,"Kepala Staf Kepresidenan (KSP) Dudung Abdurachman mengonfirmasi bahwa isu miring mengenai praktik ""jual beli titik dapur"" dalam program nasional Makan Bergizi Gratis (MBG) menjadi salah satu faktor utama yang membuat Presiden mengambil tindakan tegas mencopot Dadan Hindayana dari https://t.co/pACS0lV2jE"


2. Mengambil Kolom Untuk Dianalisa

In [5]:
# Mengambil kolom 'text' dan mengubah namanya menjadi 'full_text'
df = df_raw[['text']].copy()
df.rename(columns={'text': 'full_text'}, inplace=True)

3. Preprocessing NLP

In [6]:
# a. Cleaning (Tidak boleh ada simbol, angka, URL, dll)
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Hapus URL
    text = re.sub(r'\@\w+|\#', '', text) # Hapus mention dan hashtag
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # Hanya menyisakan huruf dan spasi
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi berlebih
    return text

df['cleaned'] = df['full_text'].apply(clean_text)

# b. Case Folding (Huruf diubah menjadi kecil semua)
df['case_folded'] = df['cleaned'].str.lower()

# c. Tokenisasi (Dipecah menjadi token/kata)
df['tokenized'] = df['case_folded'].apply(word_tokenize)

# d. Filtering (Menghapus stop words / kata hubung bahasa Indonesia)
stop_words = set(stopwords.words('indonesian'))
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['filtered'] = df['tokenized'].apply(remove_stopwords)

# e. Stemming (Diubah menjadi kata dasar)
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def apply_stemming(tokens):
    # Gabungkan token jadi kalimat, di-stem, lalu dipecah lagi jadi token
    text_joined = ' '.join(tokens)
    stemmed_text = stemmer.stem(text_joined)
    return list(stemmed_text.split())

print("Sedang melakukan stemming... (Tunggu sebentar, proses ini memakan waktu beberapa menit)")
df['stemmed'] = df['filtered'].apply(apply_stemming)

Sedang melakukan stemming... (Tunggu sebentar, proses ini memakan waktu beberapa menit)


4. Menampilkan Hasil

In [7]:
display(df.head(10))

,full_text,cleaned,case_folded,tokenized,filtered,stemmed
0,"Kejaksaan Agung menetapkan Eks Kepala Badan Gizi Nasional (BGN) Dadan Hindayana sebagai tersangka kasus dugaan penyimpangan tata kelola program Makan Bergizi Gratis (MBG). Dadan kini ditahan di Rutan Salemba.\n\nSelain Dadan, Kejagung juga menetapkan Eks Waka BGN Sonny Sonjaya dan https://t.co/OL4KIBw2Js",Kejaksaan Agung menetapkan Eks Kepala Badan Gizi Nasional BGN Dadan Hindayana sebagai tersangka kasus dugaan penyimpangan tata kelola program Makan Bergizi Gratis MBG Dadan kini ditahan di Rutan Salemba Selain Dadan Kejagung juga menetapkan Eks Waka BGN Sonny Sonjaya dan,kejaksaan agung menetapkan eks kepala badan gizi nasional bgn dadan hindayana sebagai tersangka kasus dugaan penyimpangan tata kelola program makan bergizi gratis mbg dadan kini ditahan di rutan salemba selain dadan kejagung juga menetapkan eks waka bgn sonny sonjaya dan,"[kejaksaan, agung, menetapkan, eks, kepala, badan, gizi, nasional, bgn, dadan, hindayana, sebagai, tersangka, kasus, dugaan, penyimpangan, tata, kelola, program, makan, bergizi, gratis, mbg, dadan, kini, ditahan, di, rutan, salemba, selain, dadan, kejagung, juga, menetapkan, eks, waka, bgn, sonny, sonjaya, dan]","[kejaksaan, agung, menetapkan, eks, kepala, badan, gizi, nasional, bgn, dadan, hindayana, tersangka, dugaan, penyimpangan, tata, kelola, program, makan, bergizi, gratis, mbg, dadan, ditahan, rutan, salemba, dadan, kejagung, menetapkan, eks, waka, bgn, sonny, sonjaya]","[jaksa, agung, tetap, eks, kepala, badan, gizi, nasional, bgn, dad, hindayana, sangka, duga, simpang, tata, kelola, program, makan, gizi, gratis, mbg, dad, tahan, rutan, salemba, dad, jagung, tetap, eks, waka, bgn, sonny, sonjaya]"
1,"MAKAN BERGIZI GRATIS\nMakanan nyang bergizi nakeuh hak mandum ureung, dan deungon program makan gratis nyoe, geutanyoe jeut tapeu pasti mandum ureung meurumpok nutrisi nyang dibutuhkan untuk udep sehat.#PenuhiGiziIndonesia #Indonesiaemas #AstaCita #Indonesiamaju #GiziBaik https://t.co/oLpDVbSayE",MAKAN BERGIZI GRATIS Makanan nyang bergizi nakeuh hak mandum ureung dan deungon program makan gratis nyoe geutanyoe jeut tapeu pasti mandum ureung meurumpok nutrisi nyang dibutuhkan untuk udep sehat PenuhiGiziIndonesia Indonesiaemas AstaCita Indonesiamaju GiziBaik,makan bergizi gratis makanan nyang bergizi nakeuh hak mandum ureung dan deungon program makan gratis nyoe geutanyoe jeut tapeu pasti mandum ureung meurumpok nutrisi nyang dibutuhkan untuk udep sehat penuhigiziindonesia indonesiaemas astacita indonesiamaju gizibaik,"[makan, bergizi, gratis, makanan, nyang, bergizi, nakeuh, hak, mandum, ureung, dan, deungon, program, makan, gratis, nyoe, geutanyoe, jeut, tapeu, pasti, mandum, ureung, meurumpok, nutrisi, nyang, dibutuhkan, untuk, udep, sehat, penuhigiziindonesia, indonesiaemas, astacita, indonesiamaju, gizibaik]","[makan, bergizi, gratis, makanan, nyang, bergizi, nakeuh, hak, mandum, ureung, deungon, program, makan, gratis, nyoe, geutanyoe, jeut, tapeu, mandum, ureung, meurumpok, nutrisi, nyang, dibutuhkan, udep, sehat, penuhigiziindonesia, indonesiaemas, astacita, indonesiamaju, gizibaik]","[makan, gizi, gratis, makan, nyang, gizi, nakeuh, hak, mandum, ureung, deungon, program, makan, gratis, nyoe, geutanyoe, jeut, tapeu, mandum, ureung, meurumpok, nutrisi, nyang, butuh, udep, sehat, penuhigiziindonesia, indonesiaemas, astacita, indonesiamaju, gizibaik]"
2,"Budalkan sam @JoeTirtoDiputro GA lokal malang kota, warung makan gratis, in-sya Allah buka jam 7.30 menyediakan kuang lebih 100 porsi, semoga Allah mudahkan dan jaga istiqomah kami semua. Barokallahu fiikum. \n#warungmakangratis\n#gratistanpasyarat https://t.co/SNddiK8uMa",Budalkan sam GA lokal malang kota warung makan gratis in sya Allah buka jam menyediakan kuang lebih porsi semoga Allah mudahkan dan jaga istiqomah kami semua Barokallahu fiikum warungmakangratis gratistanpasyarat,budalkan sam ga lokal malang kota warung makan gratis in sya allah buka jam menyediakan kuang 

In [8]:
pd.set_option('display.max_rows', None) # Membuka limit baris pandas
display(df)

,full_text,cleaned,case_folded,tokenized,filtered,stemmed
0,"Kejaksaan Agung menetapkan Eks Kepala Badan Gizi Nasional (BGN) Dadan Hindayana sebagai tersangka kasus dugaan penyimpangan tata kelola program Makan Bergizi Gratis (MBG). Dadan kini ditahan di Rutan Salemba.\n\nSelain Dadan, Kejagung juga menetapkan Eks Waka BGN Sonny Sonjaya dan https://t.co/OL4KIBw2Js",Kejaksaan Agung menetapkan Eks Kepala Badan Gizi Nasional BGN Dadan Hindayana sebagai tersangka kasus dugaan penyimpangan tata kelola program Makan Bergizi Gratis MBG Dadan kini ditahan di Rutan Salemba Selain Dadan Kejagung juga menetapkan Eks Waka BGN Sonny Sonjaya dan,kejaksaan agung menetapkan eks kepala badan gizi nasional bgn dadan hindayana sebagai tersangka kasus dugaan penyimpangan tata kelola program makan bergizi gratis mbg dadan kini ditahan di rutan salemba selain dadan kejagung juga menetapkan eks waka bgn sonny sonjaya dan,"[kejaksaan, agung, menetapkan, eks, kepala, badan, gizi, nasional, bgn, dadan, hindayana, sebagai, tersangka, kasus, dugaan, penyimpangan, tata, kelola, program, makan, bergizi, gratis, mbg, dadan, kini, ditahan, di, rutan, salemba, selain, dadan, kejagung, juga, menetapkan, eks, waka, bgn, sonny, sonjaya, dan]","[kejaksaan, agung, menetapkan, eks, kepala, badan, gizi, nasional, bgn, dadan, hindayana, tersangka, dugaan, penyimpangan, tata, kelola, program, makan, bergizi, gratis, mbg, dadan, ditahan, rutan, salemba, dadan, kejagung, menetapkan, eks, waka, bgn, sonny, sonjaya]","[jaksa, agung, tetap, eks, kepala, badan, gizi, nasional, bgn, dad, hindayana, sangka, duga, simpang, tata, kelola, program, makan, gizi, gratis, mbg, dad, tahan, rutan, salemba, dad, jagung, tetap, eks, waka, bgn, sonny, sonjaya]"
1,"MAKAN BERGIZI GRATIS\nMakanan nyang bergizi nakeuh hak mandum ureung, dan deungon program makan gratis nyoe, geutanyoe jeut tapeu pasti mandum ureung meurumpok nutrisi nyang dibutuhkan untuk udep sehat.#PenuhiGiziIndonesia #Indonesiaemas #AstaCita #Indonesiamaju #GiziBaik https://t.co/oLpDVbSayE",MAKAN BERGIZI GRATIS Makanan nyang bergizi nakeuh hak mandum ureung dan deungon program makan gratis nyoe geutanyoe jeut tapeu pasti mandum ureung meurumpok nutrisi nyang dibutuhkan untuk udep sehat PenuhiGiziIndonesia Indonesiaemas AstaCita Indonesiamaju GiziBaik,makan bergizi gratis makanan nyang bergizi nakeuh hak mandum ureung dan deungon program makan gratis nyoe geutanyoe jeut tapeu pasti mandum ureung meurumpok nutrisi nyang dibutuhkan untuk udep sehat penuhigiziindonesia indonesiaemas astacita indonesiamaju gizibaik,"[makan, bergizi, gratis, makanan, nyang, bergizi, nakeuh, hak, mandum, ureung, dan, deungon, program, makan, gratis, nyoe, geutanyoe, jeut, tapeu, pasti, mandum, ureung, meurumpok, nutrisi, nyang, dibutuhkan, untuk, udep, sehat, penuhigiziindonesia, indonesiaemas, astacita, indonesiamaju, gizibaik]","[makan, bergizi, gratis, makanan, nyang, bergizi, nakeuh, hak, mandum, ureung, deungon, program, makan, gratis, nyoe, geutanyoe, jeut, tapeu, mandum, ureung, meurumpok, nutrisi, nyang, dibutuhkan, udep, sehat, penuhigiziindonesia, indonesiaemas, astacita, indonesiamaju, gizibaik]","[makan, gizi, gratis, makan, nyang, gizi, nakeuh, hak, mandum, ureung, deungon, program, makan, gratis, nyoe, geutanyoe, jeut, tapeu, mandum, ureung, meurumpok, nutrisi, nyang, butuh, udep, sehat, penuhigiziindonesia, indonesiaemas, astacita, indonesiamaju, gizibaik]"
2,"Budalkan sam @JoeTirtoDiputro GA lokal malang kota, warung makan gratis, in-sya Allah buka jam 7.30 menyediakan kuang lebih 100 porsi, semoga Allah mudahkan dan jaga istiqomah kami semua. Barokallahu fiikum. \n#warungmakangratis\n#gratistanpasyarat https://t.co/SNddiK8uMa",Budalkan sam GA lokal malang kota warung makan gratis in sya Allah buka jam menyediakan kuang lebih porsi semoga Allah mudahkan dan jaga istiqomah kami semua Barokallahu fiikum warungmakangratis gratistanpasyarat,budalkan sam ga lokal malang kota warung makan gratis in sya allah buka jam menyediakan kuang 